# Character-Level Language Model with GRU

Trains a character-level recurrent network on Shakespeare text to learn to generate English sentences.  
Demonstrates RNN sequence modelling using TensorFlow/Keras GRU (CPU-compatible, no GPU required).

Original concept from: https://www.kaggle.com/anilreddy8989/word-guess-with-rnn  
Rewritten to use TF2 native APIs and a publicly downloadable dataset.

In [ ]:
import urllib.request
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from pathlib import Path

print(f'TensorFlow {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs available: {len(gpus)} ({gpus[0].name if gpus else "CPU only"})')

In [ ]:
# Download Shakespeare text (~1 MB corpus hosted by TensorFlow)
DATA_URL = 'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt'
cache_path = Path('shakespeare.txt')

if not cache_path.exists():
    print('Downloading Shakespeare corpus...')
    urllib.request.urlretrieve(DATA_URL, cache_path)
    print(f'Saved to {cache_path}')
else:
    print(f'Using cached {cache_path}')

text = cache_path.read_text(encoding='utf-8')
print(f'Corpus length : {len(text):,} characters')
print('Sample text   :', repr(text[:200]))

In [ ]:
# Build character-level vocabulary
vocab = sorted(set(text))
vocab_size = len(vocab)
print(f'Unique characters: {vocab_size}')
print('Vocabulary:', vocab)

char2idx = {c: i for i, c in enumerate(vocab)}
idx2char = np.array(vocab)

# Encode the full text as integer indices
text_encoded = np.array([char2idx[c] for c in text], dtype=np.int32)
print(f'\nEncoded array shape: {text_encoded.shape}')
print('First 20 chars :', repr(text[:20]))
print('Encoded as     :', text_encoded[:20])

In [ ]:
# Create (input, target) sequence pairs for training
SEQ_LEN     = 100     # length of each training sequence
BATCH_SIZE  = 64
BUFFER_SIZE = 10_000  # shuffle buffer

char_dataset = tf.data.Dataset.from_tensor_slices(text_encoded)
sequences    = char_dataset.batch(SEQ_LEN + 1, drop_remainder=True)

def split_input_target(chunk):
    """Offset target by 1: model learns to predict the *next* character."""
    return chunk[:-1], chunk[1:]

dataset = (
    sequences
    .map(split_input_target)
    .shuffle(BUFFER_SIZE)
    .batch(BATCH_SIZE, drop_remainder=True)
    .prefetch(tf.data.AUTOTUNE)
)

# Verify shapes
for x_batch, y_batch in dataset.take(1):
    print(f'Input  shape: {x_batch.shape}')   # (64, 100)
    print(f'Target shape: {y_batch.shape}')   # (64, 100)
    seq = x_batch[0].numpy()
    print(f'Sample input : {repr("".join(idx2char[seq[:50]]))}')
    print(f'Sample target: {repr("".join(idx2char[y_batch[0].numpy()[:50]]))}')

In [ ]:
# Build a character-level GRU model (CPU-compatible — no CuDNN required)
EMBEDDING_DIM = 128
GRU_UNITS     = 512

model = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, EMBEDDING_DIM, name='embedding'),
    tf.keras.layers.GRU(
        GRU_UNITS,
        return_sequences=True,
        reset_after=True,   # standard implementation that works on CPU and GPU
        name='gru'
    ),
    tf.keras.layers.Dense(vocab_size, name='logits')
], name='char_lm')

model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
)

model.summary()

In [ ]:
# Train the model
EPOCHS = 10

history = model.fit(dataset, epochs=EPOCHS, verbose=1)

print(f'\nInitial loss : {history.history["loss"][0]:.4f}')
print(f'Final loss   : {history.history["loss"][-1]:.4f}')

In [ ]:
# Plot training loss
plt.figure(figsize=(8, 4))
plt.plot(range(1, EPOCHS + 1), history.history['loss'], marker='o')
plt.title('Character-Level GRU Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Sparse Categorical Cross-Entropy')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Generate text from the trained model (auto-regressive sampling)
def generate_text(model, start_string, num_generate=400, temperature=1.0):
    """Generate characters one-at-a-time using stateful single-step model."""
    # Build a stateful inference copy with batch_size=1
    gen_model = tf.keras.Sequential([
        tf.keras.layers.Embedding(vocab_size, EMBEDDING_DIM,
                                  batch_input_shape=[1, None]),
        tf.keras.layers.GRU(GRU_UNITS, return_sequences=True,
                            reset_after=True, stateful=True),
        tf.keras.layers.Dense(vocab_size)
    ])
    gen_model.set_weights(model.get_weights())
    gen_model.reset_states()

    # Encode seed string (skip unknown characters)
    input_ids = tf.expand_dims([char2idx[c] for c in start_string if c in char2idx], 0)

    generated = []
    for _ in range(num_generate):
        logits  = gen_model(input_ids)           # (1, seq_len, vocab_size)
        logits  = logits[:, -1, :] / temperature # last step, apply temperature
        pred_id = tf.random.categorical(logits, num_samples=1)[0, 0].numpy()
        generated.append(idx2char[pred_id])
        input_ids = tf.expand_dims([pred_id], 0) # feed prediction back

    return start_string + ''.join(generated)

print('=== Generated text (temperature=1.0 — creative) ===')
print(generate_text(model, 'ROMEO: ', num_generate=400, temperature=1.0))
print()
print('=== Generated text (temperature=0.5 — conservative) ===')
print(generate_text(model, 'JULIET: ', num_generate=200, temperature=0.5))